In [46]:
import pandas as pd
import os
import subprocess
from PIL import Image, ImageDraw, ImageFont
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib import colors
from pathlib import Path
from pywinauto.application import Application
from pywinauto.keyboard import send_keys
from pywinauto import Desktop
import re
import pyperclip
import time
import fitz
import math

In [ ]:
brf_dir = "stories_text_BRF/"
tfiles = [ff for ff in os.listdir(brf_dir) if (ff.split(".")[-1]=="txt") and ff!="01_Boar.txt"]
overwrite = False

for ff in tfiles:
    brf_path = Path.cwd() / "stories_text_BRF" / f"{ff.split(".")[0]}.brf"
    if not overwrite and brf_path.exists():
        continue
    # print(ff)
    # continue
    app = Application(backend="uia").start("brailleblaster.exe")
    time.sleep(5)
    # win = app.window(title_re=".*BrailleBlaster.*")
    # win.wait("visible", timeout=20)
    # win.set_focus()
    send_keys("^o")
    time.sleep(1)


    # inspect once if needed
    # dlg.print_control_identifiers()

    # filename_box = dlg.child_window(title="File name:", control_type="Edit")
    file_path = Path.cwd() / "stories_text_BRF" / "01_Boar.bbz"
    pyperclip.copy(str(file_path))
    send_keys("^v")
    time.sleep(0.3)
    send_keys("{ENTER}")
    time.sleep(1)


    send_keys("^{HOME}")       # Ctrl+Home
    time.sleep(1)

    send_keys("+{END}")       # Ctrl+Shift+End
    time.sleep(1)

    for ii in range(210):
        send_keys("+{DOWN}")

    send_keys("{DEL}")
    time.sleep(1)

    with open(f"stories_text_BRF\\{ff}", "r") as f:
        txt = f.read()
    pyperclip.copy(txt)

    # send_keys("^a")
    send_keys("^v")
    time.sleep(1)
    send_keys("%fe{RIGHT}v{ENTER}")
    time.sleep(1)

    # remove brf_path if it exists
    if brf_path.exists():
        brf_path.unlink()
    pyperclip.copy(str(brf_path))
    send_keys("^v")
    time.sleep(1)
    send_keys("{ENTER}")
    time.sleep(1)

    send_keys("%{F4}")
    send_keys("n")
    send_keys("{ENTER}")
    time.sleep(5)

In [152]:
# ======= image creation params =======
DPI = 300
IN_to_PT = 72

CARD_W_IN = 8
CARD_H_IN = 5.5
CARD_W_PT = CARD_W_IN*IN_to_PT
CARD_H_PT = CARD_H_IN*IN_to_PT

BRAILLE_X_IN = 0.9
BRAILLE_Y_IN = 0.2
BRAILLE_W_IN = 6
BRAILLE_H_IN = 5.1

BORDER_W_IN = 0.05

BRL_CELL_H_PT = 0.263*IN_to_PT
LINE_GAP_PT = 0.05*IN_to_PT
# LINE_PITCH_PT = BRL_CELL_H_PT + LINE_GAP_PT
LINE_PITCH_PT = 0.4 * IN_to_PT

round_c_r = 0.15

ID_H = 1.5
ID_W = 0.52

print((CARD_W_IN-BRAILLE_W_IN)/2-ID_W*1.5)
print((CARD_H_IN-BRAILLE_H_IN)/2+BRAILLE_H_IN-ID_H)

SWELL_BRAILLE_FONT = r"swell-braille.ttf"
ID_FONT = r"C:\\Windows\\Fonts\\impact.ttf"
TEXT_FONT = r"C:\\Windows\\Fonts\\calibri.ttf"

BRAILLE_FONT_SIZE = 100
ID_FONT_SIZE_PT = 0.45 * IN_to_PT
TXT_FONT_SIZE_PT = 0.3 * IN_to_PT

pdfmetrics.registerFont(TTFont("brl", SWELL_BRAILLE_FONT))
pdfmetrics.registerFont(TTFont("id", ID_FONT))
pdfmetrics.registerFont(TTFont("txt", TEXT_FONT))
braille_font = ImageFont.truetype(SWELL_BRAILLE_FONT, size=BRAILLE_FONT_SIZE)
id_font = ImageFont.truetype(ID_FONT, size=150)
back_font = ImageFont.truetype(TEXT_FONT, size=100)

def inch(x):
    return int(round(x * DPI))

def render_front_card(brf_text, card_id, out_path):
    img = Image.new("RGB", (inch(CARD_W_IN), inch(CARD_H_IN)), "black")
    draw = ImageDraw.Draw(img)
    draw.rounded_rectangle([0, 0, inch(CARD_W_IN), inch(CARD_H_IN)], radius=inch(round_c_r), fill="white",outline="#000000", width=inch(0.005))

    x0 = inch(BRAILLE_X_IN)
    y0 = inch(BRAILLE_Y_IN)
    x1 = x0 + inch(BRAILLE_W_IN)
    y1 = y0 + inch(BRAILLE_H_IN)

    # Red border
    draw.rectangle(
        [x0, y0, x1, y1],
        outline=(220, 0, 0),
        width=inch(BORDER_W_IN)
    )

    # Braille text
    pad_x = inch(0.2)
    pad_y = inch(0.3)

    draw.multiline_text(
        (x0 + pad_x, y0 + pad_y),
        brf_text,
        font=braille_font,
        fill="black",
        spacing=40,
    )

    # Card ID, lower-left corner
    # draw the text elsewhere and rotate it 90 degrees, and paste it back on the image, since PIL doesn't support rotated text
    
    id_img = Image.new("RGBA", (inch(ID_H), inch(ID_W)), (255, 255, 255, 0))
    id_draw = ImageDraw.Draw(id_img)
    id_draw.text((0, 0), card_id, font=id_font, fill="black")
    id_img = id_img.rotate(90, expand=1)
    img.paste(id_img, (inch((CARD_W_IN-BRAILLE_W_IN)/2-ID_W*0.9), inch((CARD_H_IN-BRAILLE_H_IN)/2+BRAILLE_H_IN-ID_H)), id_img)
    
    img.save(out_path, quality=95)
    return

def wrap_text_by_pixels(text, font, max_width_px, draw):
    words = text.replace("\n", " ").split()
    lines = []
    current = ""

    for word in words:
        test = word if not current else current + " " + word
        bbox = draw.textbbox((0, 0), test, font=font)
        if bbox[2] - bbox[0] <= max_width_px:
            current = test
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return "\n".join(lines)


def render_back_card(txt, card_id, out_path):
    img = Image.new("RGB", (inch(CARD_W_IN), inch(CARD_H_IN)), "black")
    draw = ImageDraw.Draw(img)
    draw.rounded_rectangle([0, 0, inch(CARD_W_IN), inch(CARD_H_IN)], radius=inch(round_c_r), fill="white",outline="#000000")

    # Card ID / serial
    draw.text(
        (inch(0.25), inch(0.25)),
        card_id,
        font=id_font,
        fill="black"
    )

    # Text snippet
    text_x = inch(0.75)
    text_y = inch(1.0)
    text_w = inch(6.75)

    wrapped = wrap_text_by_pixels(txt, back_font, text_w, draw)

    draw.multiline_text(
        (text_x, text_y),
        wrapped,
        font=back_font,
        fill="black",
        spacing=30,
    )

    img.save(out_path, quality=95)
    return

def render_front_card_PDF(brf_text, card_id, out_path):
    c = canvas.Canvas(str(out_path), pagesize=(CARD_W_PT, CARD_H_PT))
    c.setFillColor(colors.white)
    c.setStrokeColor(colors.black)
    c.setLineWidth(0.005 * IN_to_PT)  # inches → points

    c.roundRect(0, 0, CARD_W_PT, CARD_H_PT, round_c_r * IN_to_PT, stroke=1, fill=1)

    x0 = BRAILLE_X_IN*IN_to_PT
    y0 = BRAILLE_Y_IN*IN_to_PT
    x1 = x0 + BRAILLE_W_IN*IN_to_PT
    y1 = y0 + BRAILLE_H_IN*IN_to_PT
    w = BRAILLE_W_IN*IN_to_PT
    h = BRAILLE_H_IN * IN_to_PT

    c.setStrokeColorRGB(220/255, 0, 0)
    c.setLineWidth(BORDER_W_IN * IN_to_PT)
    c.rect(x0, y0, w, h, stroke=1, fill=0)
    

    # Braille text
    pad_x = 0.5*IN_to_PT
    pad_y = 0.4*IN_to_PT

    c.setFillColor(colors.black)
    c.setFont("brl", BRL_CELL_H_PT)

    y_text = y1 - pad_y - BRL_CELL_H_PT

    for line in brf_text.split("\n"):
        c.drawString(x0 + pad_x, y_text, line)
        y_text -= LINE_PITCH_PT

    c.saveState()
    tx = ((CARD_W_IN - BRAILLE_W_IN)/2 - ID_W*0.5) * IN_to_PT
    ty = ((CARD_H_IN - BRAILLE_H_IN)/2) * IN_to_PT
    c.translate(tx, ty)
    c.rotate(90)
    c.setFont("id", ID_FONT_SIZE_PT)
    c.drawString(0, 0, card_id)
    c.restoreState()
    
    c.save()
    return

def render_back_card_PDF(txt, card_id, out_path):
    text_x = 0.5
    text_y = 4.3
    id_y = 4.75
    line_dist = 0.35

    
    c = canvas.Canvas(str(out_path), pagesize=(CARD_W_PT, CARD_H_PT))
    c.setFillColor(colors.white)
    c.setStrokeColor(colors.black)
    c.setLineWidth(0.005 * IN_to_PT) 

    c.roundRect(0, 0, CARD_W_PT, CARD_H_PT, round_c_r * IN_to_PT, stroke=1, fill=1)

    c.setFillColor(colors.black)
    c.setFont("id", ID_FONT_SIZE_PT)
    c.drawString(text_x*IN_to_PT, id_y*IN_to_PT, card_id)

    img = Image.new("RGB", (inch(CARD_W_IN), inch(CARD_H_IN)), "black")
    draw = ImageDraw.Draw(img)
    wrapped = wrap_text_by_pixels(txt, back_font, inch(7.5), draw)    

    c.setFont("txt", TXT_FONT_SIZE_PT)
    for line in wrapped.split("\n"):
        c.drawString(text_x*IN_to_PT, text_y*IN_to_PT, line)
        text_y -= line_dist

    c.save()
    return


0.21999999999999997
3.8


In [153]:
brf_dir = "stories_text_BRF/"
bfiles = [ff for ff in os.listdir(brf_dir) if ff.split(".")[-1]=="brf"]
brf_df = pd.DataFrame(columns=["story", "page", "ID", "brf", "txt"])
translator = ".\\liblouis-3.37.0-win64\\bin\\lou_translate.exe"

OUT_DIR = Path("card_images")
FRONT_DIR = OUT_DIR / "front"
BACK_DIR = OUT_DIR / "back"
# FRONT_DIR.mkdir(parents=True, exist_ok=True)
# BACK_DIR.mkdir(parents=True, exist_ok=True)

cwd = os.getcwd()

bb_to_louis = {"\\":"|", "[":"{", "]":"}", "^":"~"}

for ff in bfiles[:1]:
    story_id = ff.split("_")[0]
    brf_text = open(brf_dir+ff, "r").read()

    # print(len(brf_text.split("\f")))
    # continue
    # Each line in each page is indicated by a new line. 
    # Pages are separated by an additional new line.
    # Empty lines in the last page are still marked by newlines, so we have to strip them.
    pages = brf_text.rstrip().lower().split("\f")
    print(len(pages))
    for ii,pp in enumerate(pages):
        for kk,vv in bb_to_louis.items():
            pp = pp.replace(kk,vv)
        print(pp)
        print("========================")
        with open("tmp.txt", "w+") as tmp:
            tmp.write(pp)
        cmd = f"{translator} -b en-us-g2.ctb < tmp.txt"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True,cwd=cwd)
        txt = result.stdout

        card_id = f"{story_id}-{(ii+1):02d}"
        to_add = {"story": story_id, "page": ii+1, "ID": card_id, "brf": "### "+pp, "txt": txt}
        # front_path = FRONT_DIR / f"{card_id}_front.jpg"
        # back_path = BACK_DIR / f"{card_id}_back.jpg"
        # render_front_card(pp, card_id, front_path)
        # render_back_card(txt, card_id, back_path)

        front_path = FRONT_DIR / f"{card_id}_front.pdf"
        render_front_card_PDF(pp, card_id, front_path)
        back_path = BACK_DIR / f"{card_id}_back.pdf"
        render_back_card_PDF(txt, card_id, back_path)

        if brf_df.shape[0] == 0:
            brf_df = pd.DataFrame([to_add])
        else:
            brf_df = pd.concat([brf_df, pd.DataFrame([to_add])], ignore_index=True)

        # print(cmd)
        print("\n\n")

brf_df.to_excel("card_data.xlsx", index=False)


19
  ,if y 7 to j|rney to !
,nor? ( ,5gl&1 y wd come
to a valley t is surr.d$
by moors z hi< z m.ta9s4
,x is 9 ? valley ": y wd
f9d ! c;y ( ,brad=d1 ":
once a ?|s& sp9n+-j5nies
t humm$ & clatt}$ spun
wool 9to m"oy = !
l;g-be>d$ mill {n}s4 ,t
all mill {n}s 7 g5}ally




busy z b1v}s & q pl1s$ )
!mvs = 2+ s su3ess;l &
well (f 0 "kn to !
resid5ts ( ,brad=d1 & if
y 7 to g 9to ! c;y to
visit ! /ately ,c;y
,hall1 y wd see "! !
,cre/ ( ! ,c;y (
,brad=d1 : ~? same
mill-{n}s cr1t$ to
celebrate _! a*ieve;ts4




,x %{s a s9i/} look+
bo>'s h1d sitt+ on top (
a well : seems puzzl+ at
f/1 b ! r1son = ? symbol
is a matt} ( leg5d4
  ,"! 0 once1 leg5d has
x1 a fe>"s bo>1 : liv$ 9
a wood locat$ j |tside !
manor ( ,brad=d4 ,a
s|rce ( grt tr|ble to !
local folk ! bo> was1




br++ t}ror to ! p1ce;l
flocks & ravag+ !
c.tryside >.d4 ,ev5
worse1 h{"e1 ! bo> mo/
lik$ to g to ! well t 0
9 ! wood & dr9k xs fre%
wat}1 s t ! p ( ,brad=d
_h second ?"|s ab visit+
! well4
  ,t ! p ( ,brad=d bore
! brunt ( ! b1/'s f

## Insert files into Steve's template

In [ ]:
template_pdf = {"front":Path("card_images/braille_cards_front.pdf"),
                "back":Path("card_images/braille_cards_back.pdf")}
images_dir = {"front":Path("card_images/front"),
              "back":Path("card_images/back")}
output_dir = Path("card_images")
# output_pdf = {"front":Path("card_images/braille_front_ready.pdf"),
#               "back":Path("card_images/braille_back_ready.pdf")}

rotate = 90
inset_pt = 0

def natural_sort_key(path):
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r"(\d+)", path.name)]


def find_pdfs(folder):
    exts = {".pdf"}
    return sorted(
        [p for p in folder.iterdir() if p.suffix.lower() in exts],
        key=natural_sort_key
    )

def slot_rect(side, page_rect, slot_number, cols=10, rows=3, inset_pt=0):
    """
    side: "front" or "back"
    The template has 30 slots:
      1-10   = bottom row
      11-20  = middle row
      21-30  = top row
    """
    cell_w = page_rect.width / cols
    cell_h = page_rect.height / rows

    idx = slot_number - 1
    col = idx % cols
    if side=="back":
        col = cols - col - 1
    row_from_bottom = idx // cols
    row_from_top = rows - 1 - row_from_bottom

    x0 = page_rect.x0 + col * cell_w + inset_pt
    y0 = page_rect.y0 + row_from_top * cell_h + inset_pt
    x1 = page_rect.x0 + (col + 1) * cell_w - inset_pt
    y1 = page_rect.y0 + (row_from_top + 1) * cell_h - inset_pt

    return fitz.Rect(x0, y0, x1, y1)


def make_braille_sheet(side, template_pdf, images_dir, output_dir, rotate=270, inset_pt=0):
    image_paths = find_pdfs(images_dir[side])
    if not image_paths:
        raise ValueError(f"No image files found in {images_dir[side]}")

    template = fitz.open(template_pdf[side])
    if template.page_count != 1:
        raise ValueError("This version expects a one-page template PDF.")

    cards_per_page = 30
    n_pages = math.ceil(len(image_paths) / cards_per_page)

    for page_i in range(n_pages):
        out = fitz.open()
        out.insert_pdf(template, from_page=0, to_page=0)
        page = out[0]

        chunk = image_paths[
            page_i * cards_per_page : (page_i + 1) * cards_per_page
        ]

        for local_i, pdf_path in enumerate(chunk, start=1):
            rect = slot_rect(side, page.rect, local_i, inset_pt=inset_pt)
            card_doc = fitz.open(pdf_path)
            page.show_pdf_page(rect, card_doc, 0, rotate=rotate, keep_proportion=True, overlay=True)
            card_doc.close()
        out_path = output_dir / f"braille_{side}_{page_i+1:02d}.pdf"
        out.save(out_path, deflate=True, garbage=4)
        out.close()
    template.close()

    print(f"Found {len(image_paths)} images")
    print(f"Total pages: {n_pages}")


for side in ["front", "back"]:
    make_braille_sheet(side=side, template_pdf=template_pdf, images_dir=images_dir, output_dir=output_dir, rotate=rotate, inset_pt=inset_pt)

Found 184 images
Total pages: 7
Found 184 images
Total pages: 7
